In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Fault Labels

In [ ]:
canopy_2015 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2015_Labeled.csv")
canopy_2016 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2016_Labeled.csv")

In [ ]:
# Fault Indicator Columns
# Rather than merging label_string/label_inverter into a single priority-ordered
# column (which would hide genuine co-occurrence between fault types), we keep
# both label sets fully separate and add lightweight boolean indicators on top.
# This supports both a coarse "fault vs no-fault" detection task and full
# subsystem-level classification, without discarding information.

NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2015['has_string_fault']   = ~canopy_2015['label_string'].isin(NON_FAULT_VALUES)
canopy_2015['has_inverter_fault'] = ~canopy_2015['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2015['has_any_fault']      = canopy_2015['has_string_fault'] | canopy_2015['has_inverter_fault']
canopy_2015['fault_cooccurrence'] = canopy_2015['has_string_fault'] & canopy_2015['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2015) ===\n")
print(f"has_string_fault   : {canopy_2015['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2015['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2015['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2015['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2015.loc[canopy_2015['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2015) ===

has_string_fault   : 11146
has_inverter_fault : 10380
has_any_fault      : 20801
fault_cooccurrence : 725

As % of evaluable rows (Op_Mask=1): 10.92%


In [ ]:
NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2016['has_string_fault']   = ~canopy_2016['label_string'].isin(NON_FAULT_VALUES)
canopy_2016['has_inverter_fault'] = ~canopy_2016['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2016['has_any_fault']      = canopy_2016['has_string_fault'] | canopy_2016['has_inverter_fault']
canopy_2016['fault_cooccurrence'] = canopy_2016['has_string_fault'] & canopy_2016['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2016) ===\n")
print(f"has_string_fault   : {canopy_2016['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2016['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2016['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2016['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2016.loc[canopy_2016['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2016) ===

has_string_fault   : 7610
has_inverter_fault : 11428
has_any_fault      : 18442
fault_cooccurrence : 596

As % of evaluable rows (Op_Mask=1): 9.54%


In [ ]:
"""
Stage 1: CS-LSTM Regression
==============================
Αναφορά: "A Comprehensive Framework for Predictive Maintenance in Solar
          Photovoltaic Systems Using Contextual Deep Model"

Σκοπός:
  Πρόβλεψη της αναμενόμενης AC ισχύος (InvPAC_kW_Avg) υπό κανονικές
  συνθήκες λειτουργίας, χρησιμοποιώντας operational data + contextual
  features (εποχή, ώρα ημέρας) μέσω Kronecker product expansion.

  Το residual (measured - expected) που παράγεται εδώ χρησιμοποιείται
  ως anomaly signal στο Stage 2 (XGBoost classifier).

Μαθηματικά:
  Standard LSTM:  gate = σ(W_h·h_{t-1}  +  W_x·x_t           + b)
  CS-LSTM:        gate = σ(W_h·h_{t-1}  +  B·(x_t ⊗ Γ(z_t))  + b)

  όπου:
    x_t   = primary features (operational data) στο timestep t
    z_t   = contextual features (season, time-of-day encodings)
    Γ(z_t) = polynomial basis functions of z_t (degree 2)
    ⊗     = Kronecker product → expanded input vector

  Αυτό σημαίνει ότι τα gate weights του LSTM γίνονται functions του
  context - δηλ. το μοντέλο μαθαίνει ότι "τον Ιούλιο στις 12:00 η
  ακτινοβολία έχει διαφορετική σημασία από ό,τι τον Δεκέμβριο".
"""

# Hyperparameters (από Baruah et al. 2025 + adaptation)

Info = {
    "train_years":       [2015, 2016],
    "val_year":          2017,
    "irr_threshold":     100.0,        # W/m² - operating mask

    # Features
    "primary_features": [              # Inverter signals = x_t
        "POA_fused",                   # POA irradiance
        "AmbTemp_C_Avg",               # ambient temperature
        "module_temp",                 # module temperature
        "InvVDCin_Avg",                # DC voltage
        "InvIDCin_Avg",                # DC current
        "InvPDC_kW_Avg",               # DC power
        "InvPAC_kW_Avg_lag1",          # previous AC power
    ],
    "target":            "InvPAC_kW_Avg",
    "poly_degree":       [2,3],            # polynomial basis degree for Γ(z_t)

    # Model
    "seq_len":           [5,10,15,30,60],           # timesteps per window
    "hidden_size":       [10, 15, 20, 25, 30]       # LSTM hidden units (από Baruah et al.)
    "num_layers":        [1,2],            # LSTM layers (από Baruah et al.)
    "dropout":           [0.1, 0.2, 0.3],

    # Training
    "batch_size":        [64, 128, 256],
    "lr":                [1e-5, 1e-3],       # learning rate (από Baruah et al.)
    "epochs":            [50-150],
    "weight_decay":      1e-6,
}

# Parameters (NIST Canopy)
P_AC_RATED_KW = 260.0   # Inverter rated AC power (kW) PVP260kW
P_DC_NOM_KW   = 243.0   # DC Nameplate power (kW)

"""
Stage 2: Φορτωση μόνο normal data (no fault data)

Κρατάμε μόνο timesteps που αντιστοιχούν σε κανονική λειτουργία:

    1. Operating mask OpMask == 1: POA irradiance ≥ irr_threshold W/m²
    2. Inverter ON: InvOpStatus_Avg > 0
    3. Zero fault flags: fault_string και fault_inverter = false
    4. Positive power: InvPAC_kW_Avg > 0
    5. PR ανάμεσα σε 1.05 και 0.7

Αυτά τα data λειτουργούν ως expected-behavior baseline για εκπαίδευση του CS-LSTM.
    """

"""
Stage 3 — Contextual Features

    Δημιουργία contextual features z_t:

    Season (εποχή):
      sin_doy = sin(2π × day_of_year / 365)
      cos_doy = cos(2π × day_of_year / 365)
      → Circular encoding που αποφεύγει την ασυνέχεια Dec→Jan

    Time of day (ώρα ημέρας):
      sin_hour = sin(2π × (hour + minute/60) / 24)
      cos_hour = cos(2π × (hour + minute/60) / 24)
      → Circular encoding (23:59 ≈ 00:00)

    z_t που CS-LSTM που χρησιμοποιούνται στο polynomial basis Γ(z_t).
    """

"""
Stage 4:
    Υπολογισμός του Γ(z_t) — polynomial basis functions of z_t.

    Input:
      z      : Τα contextual features (sin_doy, cos_doy, sin_hour, cos_hour)
      degree : polynomial degree 2

    Output:
      basis  :  Πολυωνυμικά γινόμενα μέχρι βαθμού 2

    Παράδειγμα για degree=2, z=[z1, z2]:
      Γ(z) = [1, z1, z2, z1², z1*z2, z2²]

    """

"""Stage 5:

    Υπολογισμός του Kronecker product x_t ⊗ Γ(z_t).

    Input:
      x_t
      Γ(z_t)

    Μαθηματικά:
      [x1, x2] ⊗ [γ1, γ2, γ3] = [x1*γ1, x1*γ2, x1*γ3, x2*γ1, x2*γ2, x2*γ3]

    Αυτό επιτρέπει στο LSTM να έχει ξεχωριστά βαρη για κάθε combination δηλαδή
    η ακτινοβολία έχει διαφορετικό weight ανάλογα με την εποχή και ώρα.
    """

    """
    Stage 5:
    Context-Sensitive LSTM Regression Model

    Αρχιτεκτονική:
      1. Input: x_expanded = x_t ⊗ Γ(z_t)
      2. LSTM layers: εκμάθηση temporal dependencies
      3. FC output layer: πρόβλεψη InvPAC_kW_Avg

    Το μοντέλο εκπαιδεύεται μόνο σε confirmed-normal data.
    Κατά inference, τρέχει σε ολα τα data (normal + faulty)
    για να παράξει residuals.
    """

    """
    Stage 6: Evaluation

    Χρησιμοποιούμε μετρικές όπως RMSE, NRMSE για να κάνουμε την ελαχιστοποιήση των residuals μεταξύ προβλεπόμενης InvPAC_kw_Avg και predicted InvPAC_kW_Avg
    """
